In [1]:
# Cell 1a
import sys
sys.stderr = open("mcp_server.log", "w")

In [2]:
# Cell 1b
##import sys

##MCP_DIAGNOSTIC_LOG = open("mcp_server.log", "w", encoding="utf-8")
##sys.stderr = MCP_DIAGNOSTIC_LOG
##print("MCP server diagnostics will be written to mcp_server.log")

In [3]:
import os
from google.colab import userdata

if not os.environ.get("GROQ_API_KEY"):
    try:
        secret = userdata.get("GROQ_API_KEY")
        if secret:
            os.environ["GROQ_API_KEY"] = secret
    except Exception as exc:
        print("Could not read Colab secret:", type(exc).__name__)

assert os.environ.get("GROQ_API_KEY"), (
    "Enable GROQ_API_KEY in Colab Secrets before continuing."
)
print("Groq credentials loaded without displaying the key.")

Groq credentials loaded without displaying the key.


In [4]:
# Cell 1c
!pip install -q mcp "langchain[mcp]>=1.4.0,<2" langchain-groq groq

In [5]:
# Cell 2
from groq import Groq

GROQ_CLIENT = Groq(api_key=os.environ["GROQ_API_KEY"])
# GROQ_MODEL = "qwen/qwen3.8-27b"
GROQ_MODEL = "openai/gpt-oss-120b"

available_ids = sorted(m.id for m in GROQ_CLIENT.models.list().data)
print("Selected:", GROQ_MODEL)
print("Available:", GROQ_MODEL in available_ids)

if GROQ_MODEL not in available_ids:
    print("Account model IDs:", available_ids)
    raise RuntimeError(
        "Choose an available tool-capable Groq model, then rerun this cell."
    )

Selected: openai/gpt-oss-120b
Available: True


In [6]:
# Cell 3
%%writefile trip_server.py

from mcp.server import MCPServer

mcp = MCPServer("TripUtilities")

@mcp.tool()
def estimate_daily_budget(
    days: int, daily_cost: float
) -> float:
    if days < 0 or daily_cost < 0:
        raise ValueError("Inputs must be nonnegative")
    return days * daily_cost

@mcp.resource("trip://notes/helsinki")
def helsinki_notes() -> str:
    """Return illustrative planning notes for a city visit."""
    return (
        "Illustrative trip notes (not live travel data).\n"
        "City: Helsinki\n"
        "Interests: museums, architecture, and cafés.\n"
        "Preference: walk between nearby places where practical.\n"
        "Reminder: verify opening hours, accessibility, and routes."
    )

@mcp.prompt()
def outline_day_trip(city: str, focus: str = "museums") -> str:
    """Provide a reusable request for a one-day itinerary."""
    return (
        f"Prepare a one-day itinerary for {city}, focusing on {focus}. "
        "Use verified sources for opening hours and travel times; "
        "label unverified details as uncertain."
    )

if __name__ == "__main__":
    mcp.run(transport="stdio")


Overwriting trip_server.py


In [7]:
# Cell 4: Trip MCP Tool + LangChain Agent + Groq LLM

from pathlib import Path
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.mcp import MCPAdapter

# 1. Initialize LLM
chat_model = ChatGroq(
    model=GROQ_MODEL,
    temperature=0
)

# 2. Connect to MCP server and convert tools
async with MCPAdapter(Path("trip_server.py").resolve()) as adapter:

    trip_tools = await adapter.list_tools()

    print("=== LANGCHAIN TOOLS ===")
    for tool in trip_tools:
        print(tool.name, tool.description)

    # Select budget tool
    budget_tools = [
        t for t in trip_tools
        if t.name == "estimate_daily_budget"
    ]
    assert len(budget_tools) == 1

    # 3. Create LangChain agent with MCP tool
    agent = create_agent(
        model=chat_model,
        tools=budget_tools,
        system_prompt=(
            "You are a helpful trip-planning assistant. "
            "Use the estimate_daily_budget tool for "
            "budget calculations. Do not invent prices."
        )
    )

    # 4. Ask LLM to use MCP tool
    result = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Use the estimate_daily_budget tool "
                        "to calculate the total budget for "
                        "a 5-day trip costing 120 euros per day."
                    )
                }
            ]
        },
        config={"recursion_limit": 16}
    )

    # 5. Display conversation and tool calls
    print("\n=== AGENT CONVERSATION ===")
    for message in result["messages"]:
        print(
            type(message).__name__,
            str(message.content)[:700]
        )
        if getattr(message, "tool_calls", None):
            print("TOOL CALLS:", message.tool_calls)


=== LANGCHAIN TOOLS ===
estimate_daily_budget 

=== AGENT CONVERSATION ===
HumanMessage Use the estimate_daily_budget tool to calculate the total budget for a 5-day trip costing 120 euros per day.
AIMessage 
TOOL CALLS: [{'name': 'estimate_daily_budget', 'args': {'daily_cost': 120, 'days': 5}, 'id': 'fc_3385b837-a35a-4f2d-a89f-3f4b861f2a58', 'type': 'tool_call'}]
ToolMessage [{'type': 'text', 'text': '600.0', 'id': 'lc_eb3dc96a-075e-46ad-b7f4-ed23f9372f1f'}]
AIMessage The total budget for a 5‑day trip at €120 per day is **€600**.


In [8]:
# Cell 5a:  MCP resources with LangChain
# Unlike MCP tools, which the LangChain adapter converts automatically,
# here we explicitly retrieve MCP resource content and wrap it in a LangChain Document.

import sys
from pathlib import Path

from mcp import Client, StdioServerParameters
from mcp.types import TextResourceContents
from langchain_core.documents import Document

TRIP_CONFIG = StdioServerParameters(
    command=sys.executable,
    args=[str(Path("trip_server.py").resolve())]
)

async with Client(TRIP_CONFIG) as client:

    # 1. Discover MCP resources
    resources = await client.list_resources()

    print("=== AVAILABLE RESOURCES ===")
    for resource in resources.resources:
        print("NAME:", resource.name)
        print("URI:", resource.uri)

    # 2. Read and convert to LangChain Documents
    trip_documents = []

    for resource in resources.resources:
        result = await client.read_resource(str(resource.uri))

        for content in result.contents:
            if isinstance(content, TextResourceContents):
                doc = Document(
                    page_content=content.text,
                    metadata={"source": str(resource.uri)}
                )
                trip_documents.append(doc)

    # 3. Inspect LangChain Documents
    print("\n=== LANGCHAIN DOCUMENTS ===")

    for doc in trip_documents:
        print("CLASS:", type(doc).__name__)
        print("SOURCE:", doc.metadata["source"])
        print("CONTENT:\n", doc.page_content)


=== AVAILABLE RESOURCES ===
NAME: helsinki_notes
URI: trip://notes/helsinki

=== LANGCHAIN DOCUMENTS ===
CLASS: Document
SOURCE: trip://notes/helsinki
CONTENT:
 Illustrative trip notes (not live travel data).
City: Helsinki
Interests: museums, architecture, and cafés.
Preference: walk between nearby places where practical.
Reminder: verify opening hours, accessibility, and routes.


In [9]:
# Cell 5b: MCP Resources + LangChain + Groq LLM

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize LLM
chat_model = ChatGroq(
    model=GROQ_MODEL,
    temperature=0
)

# 2. Get content from LangChain Documents
assert trip_documents, "No MCP resources were loaded"

trip_context = "\n\n".join(
    doc.page_content for doc in trip_documents
)

# 3. Create LangChain prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a cautious trip-planning assistant. "
        "Use the provided resource context to answer questions. "
        "The notes are illustrative, not live travel data. "
        "Do not invent opening hours, routes, prices, "
        "or other unverified details."
    ),
    (
        "human",
        "Resource context:\n{context}\n\n"
        "Question: {question}"
    )
])

# 4. Create LangChain chain
chain = prompt | chat_model

# 5. Ask the LLM using MCP resource content
result = await chain.ainvoke({
    "context": trip_context,
    "question": (
        "What interests and preferences should guide "
        "my visit to Helsinki?"
    )
})

print("=== LLM ANSWER ===")
print(result.content)


=== LLM ANSWER ===
Based on the information you provided, your Helsinki itinerary should be shaped by the following interests and preferences:

**Key Interests**
1. **Museums** – Prioritize museums that showcase Finnish art, design, history, or natural science.
2. **Architecture** – Focus on both historic landmarks (e.g., neoclassical and Art Nouveau buildings) and contemporary works (modern public spaces, innovative residential projects, and notable works by Finnish architects such as Alvar Aalto).
3. **Cafés** – Include stops at local cafés that highlight Helsinki’s coffee culture, cozy interiors, and perhaps a view of the city’s streets or waterfront.

**Travel Preferences**
- **Walkable Distances** – Whenever possible, group attractions that are close to one another so you can explore on foot. This will let you soak up the city’s atmosphere, discover hidden details in the architecture, and enjoy spontaneous café breaks.
- **Verification Checklist** – Before you set out each day, do

In [10]:
# Cell 6a:  MCP prompts with LangChain
# Unlike MCP tools, which the LangChain adapter converts automatically,
# here we explicitly retrieve MCP prompts content and convert to LangChain ChatPromptTemplate

import sys
from pathlib import Path

from mcp import Client, StdioServerParameters
from mcp.types import TextContent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

TRIP_CONFIG = StdioServerParameters(
    command=sys.executable,
    args=[str(Path("trip_server.py").resolve())]
)

async with Client(TRIP_CONFIG) as client:

    # 1. Discover MCP prompts
    prompts = await client.list_prompts()

    print("=== AVAILABLE PROMPTS ===")
    for prompt in prompts.prompts:
        print("NAME:", prompt.name)
        print("DESCRIPTION:", prompt.description)
        print("ARGUMENTS:", prompt.arguments)

    # 2. Retrieve a rendered MCP prompt
    result = await client.get_prompt(
        "outline_day_trip",
        {"city": "Helsinki", "focus": "museums"}
    )

    # 3. Convert MCP messages to LangChain messages
    lc_messages = []

    for msg in result.messages:
        if isinstance(msg.content, TextContent):
            if msg.role == "user":
                lc_messages.append(
                    HumanMessage(content=msg.content.text)
                )
            elif msg.role == "assistant":
                lc_messages.append(
                    AIMessage(content=msg.content.text)
                )

    # 4. Convert into a LangChain ChatPromptTemplate
    trip_prompt = ChatPromptTemplate.from_messages(
        lc_messages
    )

    # 5. Inspect the LangChain prompt
    print("\n=== LANGCHAIN PROMPT ===")
    print("CLASS:", type(trip_prompt).__name__)

    for msg in trip_prompt.format_messages():
        print("ROLE:", msg.type)
        print("CONTENT:", msg.content)



=== AVAILABLE PROMPTS ===
NAME: outline_day_trip
DESCRIPTION: Provide a reusable request for a one-day itinerary.
ARGUMENTS: [PromptArgument(name='city', title=None, description=None, required=True), PromptArgument(name='focus', title=None, description=None, required=False)]

=== LANGCHAIN PROMPT ===
CLASS: ChatPromptTemplate
ROLE: human
CONTENT: Prepare a one-day itinerary for Helsinki, focusing on museums. Use verified sources for opening hours and travel times; label unverified details as uncertain.


In [11]:
# Cell 6b: MCP Prompt + LangChain + Groq LLM

from langchain_groq import ChatGroq

# 1. Initialize LLM
chat_model = ChatGroq(
    model=GROQ_MODEL,
    temperature=0
)

# 2. Create LangChain chain
chain = trip_prompt | chat_model

# 3. Invoke LLM using the MCP prompt
result = await chain.ainvoke({})

# 4. Display response
print("=== LLM ANSWER ===")
print(result.content)


=== LLM ANSWER ===
**One‑Day Museum‑Focused Itinerary – Helsinki**  
*(Suggested day: **Tuesday** – all museums listed are open on this day. Adjust the day if you travel on a Monday, when several institutions are closed.)*  

| Time (local) | Activity | Address | Opening Hours* | Travel (≈) | Notes |
|--------------|----------|---------|----------------|------------|-------|
| **09:00 – 09:30** | **Breakfast** at **Café Regatta** (iconic seaside café) | Merikannontie 23, 00260 Helsinki | – | – | Quick bite before the museums; walk to the first venue (≈10 min). |
| **09:40 – 10:55** | **Ateneum – Finnish National Gallery** (classical Finnish art) | Kaivokatu 2, 00100 Helsinki | **Tue–Sun 10:00‑18:00** (closed Mon)【1】 | **≈10 min walk** (≈800 m) | Arrive a few minutes before opening to avoid the first‑hour rush. |
| **11:00 – 12:15** | **Kiasma – Museum of Contemporary Art** | Mannerheimintie 2, 00100 Helsinki | **Tue–Sun 10:00‑18:00**【2】 | **≈7 min walk** (≈600 m) | Short walk across th